In [1]:
import httpx
import time
import pandas as pd
import re
from transformers import pipeline
from datetime import datetime, timezone
import json
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm

/Users/mnatali/Projects/sentiment_analysis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.9.1 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0610 14:52:06.885000 79542 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
file_path = Path("/Users/mnatali/Projects/sentiment_analysis/brightdata_across_social_media_analysis/brightdata_social_exports/x_datacenters_posts.json")

with file_path.open("r", encoding="utf-8") as f:
    x_posts = json.load(f)

In [3]:
all_post_ids = []

env_post_ids = []
infr_post_ids = []
housing_post_ids = []
econ_post_ids = []
life_qual_post_ids = []
aesth_post_ids = []
gov_post_ids = []
not_useful_post_ids = []

environmental_keywords = ["environment", "pollution", "emissions", "diesel", "fumes", "light pollution", "water", "drought", "water consumption",  "waste", "carbon", "climate"]
infrastructure_utilities_keywords = ["utilities", "utility", "electric", "power", "grid", "blackout", "substation", "reliable", "reliability", "capacity", "infrastructure", "generator", "water"]
housing_keywords = ["housing", "house", "home", "rent", "property", "real estate", "house price", "housing price", "housing cost", "land cost", "land price", "gentrification"]
economic_keywords = ["job", "work", "revenue", "tax", "money", "rich", "salary", "economy", "investment", "business"]
life_quality_keywords = ["resident", "fumes", "noise", "loud", "quiet", "vibration", "light pollution", "sound", "smell", "traffic"]
aesthetics_keywords = ["landscape", "visual impact",  "aesthetics", "looks", "architecture", "design", "facade", "industrial", "windows", "dark"]
government_keywords = ["zoning", "zone", "planning", "approval", "permit", "city council", "bill", "mayor", "governor", "government"]

matched_posts = 0

for x_post in x_posts:
    post_id = x_post["id"]
    all_post_ids.append(post_id)
    text = (x_post["description"])

    matched = False

    if any(kw in text for kw in environmental_keywords):
        env_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in infrastructure_utilities_keywords):
        infr_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in housing_keywords):
        housing_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in economic_keywords):
        econ_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in life_quality_keywords):
        life_qual_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in aesthetics_keywords):
        aesth_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in government_keywords):
        gov_post_ids.append(post_id)
        matched = True
    if not matched:
        not_useful_post_ids.append(post_id)
    else:
        matched_posts += 1

print("Total posts:", len(all_post_ids))
print("Total posts with a theme:", matched_posts)
print("Found environmental posts:", len(env_post_ids))
print("Found infrastructure posts:", len(infr_post_ids))
print("Found housing posts:", len(housing_post_ids))
print("Found economic posts:", len(econ_post_ids))
print("Found life quality posts:", len(life_qual_post_ids))
print("Found aesthetic posts:", len(aesth_post_ids))
print("Found government posts:", len(gov_post_ids))

posts_by_id = {post["id"]: post for post in x_posts}

Total posts: 7300
Total posts with a theme: 5305
Found environmental posts: 761
Found infrastructure posts: 3311
Found housing posts: 1113
Found economic posts: 2584
Found life quality posts: 1379
Found aesthetic posts: 912
Found government posts: 1487


In [ ]:
theme_lists = [env_post_ids, infr_post_ids, housing_post_ids, econ_post_ids, life_qual_post_ids, aesth_post_ids, gov_post_ids]
posts = pd.DataFrame(columns=["ids", "text", "date", "likes", "number of replies", "number of reposts", "number of views", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"])
datacenters_keywords = ["datacenter", "data center", "datacentre", "data centre"]

for theme in theme_lists:
    df1 = pd.DataFrame(columns=["ids", "text", "date", "likes", "number of replies", "number of reposts", "number of views", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"])
    post_ids = []
    post_texts = []
    post_dates = []
    post_likes = []
    post_num_replies = []
    post_num_reposts = []
    post_views = []


    for pid in theme:
        post_ids.append(pid)
        post = posts_by_id.get(pid)
        post_texts.append(post["description"])
        post_dates.append(post["date_posted"])
        post_likes.append(post["likes"])
        post_num_replies.append(post["replies"])
        post_num_reposts.append(post["reposts"])
        if(post["views"] == None):
            post_views.append(0)
        else:
            post_views.append(post["views"])


    df1["ids"] = post_ids
    df1["text"] = post_texts
    df1["date"] = post_dates
    df1["likes"] = post_likes
    df1["number of replies"] = post_num_replies
    df1["number of reposts"] = post_num_reposts
    df1["number of views"] = post_views


    for col in ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"]:
        df1[col] = False

    if theme == env_post_ids:
        df1["environment"] = True
    if theme == infr_post_ids:
        df1["infrastructure"] = True
    if theme == housing_post_ids:
        df1["housing"] = True
    if theme == econ_post_ids:
        df1["economy"] = True
    if theme == life_qual_post_ids:
        df1["life quality"] = True
    if theme == aesth_post_ids:
        df1["aesthetics"] = True
    if theme == gov_post_ids:
        df1["government"] = True
    posts = pd.concat([posts, df1], ignore_index=True)


posts = posts.astype({
    "ids": "string",
    "text": "string",
    "date": "string",
    "likes": "int64",
    "number of replies": "int64",
    "number of reposts": "int64",
    "number of views": "int64",
    "environment": "bool",
    "infrastructure": "bool",
    "housing": "bool",
    "economy": "bool",
    "life quality": "bool",
    "aesthetics": "bool",
    "government": "bool",
})

len(posts['ids'])

11547

In [ ]:
def remove_links(text):
    url_pattern = re.compile(r'\[https?://\S+\]|\(https?://\S+\)|\[www\.\S+\]|\(www\.\S+\)')
    cleaned_text = url_pattern.sub('', text)
    return cleaned_text

sample_string = "The board’s land use policy committee heard a presentation from the Department of Planning and Development about potential enhancements to data center use standards, including requiring a noise study and establishing a minimum distance from residential areas. The board had directed staff last May to provide research, findings and recommendations...  McKay said this process is designed to steer data centers into places that are most appropriate and make sure standards are set for environmental protections, noise mitigation and other issues that have people concerned. He said the county is learning from Loudoun and Prince William Counties, where data centers are becoming more and more prevalent...  [https://wjla.com/news/local/fairfax-county-considers-enhancing-data-center-guidelines-virginia-community-development-board-of-supervisors#](https://wjla.com/news/local/fairfax-county-considers-enhancing-data-center-guidelines-virginia-community-development-board-of-supervisors#)  A group of residents from the [Save Bren Mar From Data Center](https://sites.google.com/view/savebrenmar/home) group from the Mason District held up signs throughout Tuesday's legislative policy committee meeting, calling calling on the Fairfax county Supervisors to end by right data center development on commercial and industrial properties.  [https://patch.com/virginia/reston/data-center-zoning-policy-be-updated-fairfax-county-board](https://patch.com/virginia/reston/data-center-zoning-policy-be-updated-fairfax-county-board)Fairfax County considers enhancing data center guidelines"
result = remove_links(sample_string)
print(result)

posts["text"] = posts["text"].apply(remove_links)

The board’s land use policy committee heard a presentation from the Department of Planning and Development about potential enhancements to data center use standards, including requiring a noise study and establishing a minimum distance from residential areas. The board had directed staff last May to provide research, findings and recommendations...  McKay said this process is designed to steer data centers into places that are most appropriate and make sure standards are set for environmental protections, noise mitigation and other issues that have people concerned. He said the county is learning from Loudoun and Prince William Counties, where data centers are becoming more and more prevalent...    A group of residents from the [Save Bren Mar From Data Center] group from the Mason District held up signs throughout Tuesday's legislative policy committee meeting, calling calling on the Fairfax county Supervisors to end by right data center development on commercial and industrial propert

In [6]:
grouping_cols = ["ids", "text", "date"]
posts = posts.groupby(grouping_cols, as_index=False).max()
theme_cols = ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"]
len(posts['ids'])

5305

In [7]:
company_terms = ["AWS","Amazon","Google","Microsoft","Azure","Meta","Oracle","Equinix","Digital Realty","IBM","Facebook","Apple","QTS","Vantage","CyrusOne","CoreSite"]

for c in company_terms:
    posts[c] = posts["text"].str.contains(rf'\b{re.escape(c)}\b', case=False, na=False)

posts_with_company = posts[posts[company_terms].any(axis=1)]
posts_with_company.head()

,ids,text,date,likes,number of replies,number of reposts,number of views,environment,infrastructure,housing,...,Oracle,Equinix,Digital Realty,IBM,Facebook,Apple,QTS,Vantage,CyrusOne,CoreSite
20,1392923288725344258,SpaceX will install ground stations within Goo...,2021-05-13T19:22:45.000Z,2846,67,379,0,False,False,False,...,False,False,False,False,False,False,False,False,False,False
22,1406229062444453889,"AWS, Microsoft et al data centre cooling...jus...",2021-06-19T12:35:09.000Z,410,20,102,0,False,True,False,...,False,False,False,False,False,False,False,False,False,False
73,1634917167832367106,Happy #DaylightSavingTime from the IBM Cloud D...,2023-03-12T14:00:02.000Z,146,5,30,19363,False,False,False,...,False,False,False,True,False,False,False,False,False,False
80,1657167270211092482,Drawing from research and production teams acr...,2023-05-12T23:34:00.000Z,20,3,7,12340,False,False,False,...,False,False,False,False,False,False,False,False,False,False
92,1680566426770309123,Good primer on the impact of AI on both new an...,2023-07-16T13:13:54.000Z,160,10,33,63790,False,True,False,...,False,False,False,False,False,False,False,False,False,False


In [8]:
posts.head()

,ids,text,date,likes,number of replies,number of reposts,number of views,environment,infrastructure,housing,...,Oracle,Equinix,Digital Realty,IBM,Facebook,Apple,QTS,Vantage,CyrusOne,CoreSite
0,1004232305760538624,Introducing Project Natick Northern Isles. So ...,2018-06-06T05:23:34.000Z,7,0,3,0,True,True,False,...,False,False,False,False,False,False,False,False,False,False
1,1036922187406618624,Currently we are experiencing a downtime due t...,2018-09-04T10:21:29.000Z,16,11,0,0,False,False,True,...,False,False,False,False,False,False,False,False,False,False
2,1036998447893766145,The website is down due to issues with the coo...,2018-09-04T15:24:31.000Z,14,12,1,0,False,True,False,...,False,False,False,False,False,False,False,False,False,False
3,1062432159317852160,Some of the presentation highlights of Day 1. ...,2018-11-13T19:49:00.000Z,0,0,0,0,True,False,False,...,False,False,False,False,False,False,False,False,False,False
4,1159219919721865217,AMD is proud to announce @Twitter has deployed...,2019-08-07T21:49:00.000Z,576,15,108,0,False,True,False,...,False,False,False,False,False,False,False,False,False,False


In [ ]:
classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,
    padding=True,
    max_length=512,
    top_k=None
)

tokenizer = classifier.tokenizer

def chunk_text(text, max_tokens=450):
    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False
    )

    if len(token_ids) == 0:
        return [""]

    chunks = []

    for i in range(0, len(token_ids), max_tokens):
        chunk_ids = token_ids[i:i + max_tokens]
        chunk = tokenizer.decode(chunk_ids, skip_special_tokens=True)
        chunks.append(chunk)

    return chunks


def aggregate_chunk_results(chunk_results):
    scores = {
        "negative": [],
        "neutral": [],
        "positive": []
    }

    for result in chunk_results:
        for item in result:
            label = item["label"].lower()
            scores[label].append(item["score"])

    avg_scores = {
        label: float(np.mean(values)) if values else 0.0
        for label, values in scores.items()
    }

    final_label = max(avg_scores, key=avg_scores.get)

    return {
        "label": final_label,
        "score": avg_scores[final_label],
        "scores": avg_scores,
        "n_chunks": len(chunk_results)
    }

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [10]:
texts= posts["text"].fillna("").astype(str).tolist()
all_post_results = []

for text in tqdm(texts):
    chunks = chunk_text(text)

    chunk_results = classifier(
        chunks,
        truncation=True,
        padding=True,
        max_length=512,
        batch_size=32
    )

    post_result = aggregate_chunk_results(chunk_results)
    all_post_results.append(post_result)

posts["roberta"] = all_post_results

100%|██████████| 5305/5305 [06:47<00:00, 13.01it/s]


In [11]:
posts["name"] = posts["roberta"].apply(lambda x: x["label"])
posts["label"] = posts["name"]

posts["degree"] = posts["roberta"].apply(lambda x: x["score"])
posts = posts.drop(["roberta", "name"], axis=1)
posts.head(15)

,ids,text,date,likes,number of replies,number of reposts,number of views,environment,infrastructure,housing,...,Digital Realty,IBM,Facebook,Apple,QTS,Vantage,CyrusOne,CoreSite,label,degree
0,1004232305760538624,Introducing Project Natick Northern Isles. So ...,2018-06-06T05:23:34.000Z,7,0,3,0,True,True,False,...,False,False,False,False,False,False,False,False,positive,0.966322
1,1036922187406618624,Currently we are experiencing a downtime due t...,2018-09-04T10:21:29.000Z,16,11,0,0,False,False,True,...,False,False,False,False,False,False,False,False,negative,0.771816
2,1036998447893766145,The website is down due to issues with the coo...,2018-09-04T15:24:31.000Z,14,12,1,0,False,True,False,...,False,False,False,False,False,False,False,False,negative,0.797949
3,1062432159317852160,Some of the presentation highlights of Day 1. ...,2018-11-13T19:49:00.000Z,0,0,0,0,True,False,False,...,False,False,False,False,False,False,False,False,positive,0.553836
4,1159219919721865217,AMD is proud to announce @Twitter has deployed...,2019-08-07T21:49:00.000Z,576,15,108,0,False,True,False,...,False,False,False,False,False,False,False,False,positive,0.954027
5,1169171020202332160,Sustainable #datacenters? 💻 That’s a reality w...,2019-09-04T08:51:08.000Z,46,13,14,0,False,False,False,...,False,False,False,False,False,False,False,False,positive,0.885922
6,1177229155311616000,SAP Data Intelligence platform powered by Cisc...,2019-09-26T14:31:17.000Z,0,0,0,0,False,True,False,...,False,False,False,False,False,False,False,False,positive,0.964160
7,1214304634522632192,Experts warn that the phenomenal growth of int...,2020-01-06T21:56:00.000Z,126,16,95,0,True,False,False,...,False,False,False,False,False,False,False,False,negative,0.629782
8,1224633091685261313,"All our factories, offices, R&amp;D facilities...",2020-02-04T09:57:36.000Z,7084,72,711,0,False,True,True,...,False,False,False,False,False,False,False,False,positive,0.835785
9,1242829230699921409,"On #IntlDataCenterDay, we'd like to reflect an...",2020-03-25T15:02:33.000Z,10,0,12,0,False,False,False,...,False,False,False,False,False,False,False,False,positive,0.960358


In [12]:
# calculating average sentiment based on theme:
# Weigh all posts by their degree in the numerator and denominator, means that the average sentiment will just be +/- 1 if there are only positive or negative themes but other than that does a pretty good job of weighing neutrality
def avg_sentiment_calculation(theme):
    theme_posts = posts[posts[theme] == True]
    pos = theme_posts.loc[theme_posts["label"] == "positive", "degree"].sum()
    neg = theme_posts.loc[theme_posts["label"] == "negative", "degree"].sum()
    total = theme_posts["degree"].sum()
    return (pos-neg)/total

print("Average sentiment of environmental posts:", avg_sentiment_calculation("environment"))
print("Average sentiment of infrastructural posts:", avg_sentiment_calculation("infrastructure"))
print("Average sentiment of housing-related posts:", avg_sentiment_calculation("housing"))
print("Average sentiment of economic posts:", avg_sentiment_calculation("economy"))
print("Average sentiment of life-quality-related posts:", avg_sentiment_calculation("life quality"))
print("Average sentiment of aesthetics-related posts:", avg_sentiment_calculation("aesthetics"))
print("Average sentiment of governmental posts:", avg_sentiment_calculation("government"))

Average sentiment of environmental posts: 0.18923789122830412
Average sentiment of infrastructural posts: 0.3467689206011438
Average sentiment of housing-related posts: 0.18747474734285136
Average sentiment of economic posts: 0.3688547066407529
Average sentiment of life-quality-related posts: 0.36609521296615805
Average sentiment of aesthetics-related posts: 0.349858214808809
Average sentiment of governmental posts: 0.2024414212785815


In [13]:
## total average sentiment calculation

pos = posts.loc[posts["label"] == "positive", "degree"].sum()
neg = posts.loc[posts["label"] == "negative", "degree"].sum()
total = posts["degree"].sum()
print("Average sentiment for all of x: ", (pos-neg)/total)

Average sentiment for all of x:  0.3361941426335211


In [14]:
pos = posts.loc[posts["label"] == "positive"]
neg = posts.loc[posts["label"] == "negative"]
neutral = posts.loc[posts["label"] == "neutral"]

print("Percent of posts that are neutral:", len(neutral)/len(posts))
print("Percent of posts that are negative:", len(neg)/len(posts))
print("Percent of posts that are positive:", len(pos)/len(posts))

really_pos = pos.loc[posts["degree"] > 0.70]
really_neg = neg.loc[posts["degree"] > 0.70]

if len(neg) != 0:
    print("Percent of negative posts that are really negative:", len(really_neg)/len(neg))
if len(pos) != 0:
    print("Percent of positive posts that are really positive:", len(really_pos)/len(pos))

Percent of posts that are neutral: 0.5581526861451461
Percent of posts that are negative: 0.06352497643732329
Percent of posts that are positive: 0.37832233741753063
Percent of negative posts that are really negative: 0.32344213649851633
Percent of positive posts that are really positive: 0.6013951170901843


In [ ]:
def avg_sentiment_calculation(dataset):
    pos = dataset.loc[dataset["label"] == "positive", "degree"].sum()
    neg = dataset.loc[dataset["label"] == "negative", "degree"].sum()
    total = dataset["degree"].sum()
    if total != 0:
        return (pos-neg)/total
    else:
        return 0

posts['year'] = pd.to_datetime(posts['date']).dt.year

year_datasets = {year: posts[posts['year'] == year] for year in range(2010, 2027)}


for i in range(2010, 2027):
    print(f"Number of posts from {i}: ", len(year_datasets[i]))
    if(avg_sentiment_calculation(year_datasets[i]) != 0):
        print(f"Average sentiment for x posts from {i}: ", avg_sentiment_calculation(year_datasets[i]))

Number of posts from 2010:  0
Number of posts from 2011:  0
Number of posts from 2012:  0
Number of posts from 2013:  0
Number of posts from 2014:  0
Number of posts from 2015:  2
Average sentiment for x posts from 2015:  0.4687714041402807
Number of posts from 2016:  2
Number of posts from 2017:  6
Average sentiment for x posts from 2017:  0.3988771628828519
Number of posts from 2018:  6
Average sentiment for x posts from 2018:  0.31424451106147344
Number of posts from 2019:  3
Average sentiment for x posts from 2019:  1.0
Number of posts from 2020:  10
Average sentiment for x posts from 2020:  0.36976331260396333
Number of posts from 2021:  17
Average sentiment for x posts from 2021:  0.622539282153555
Number of posts from 2022:  30
Average sentiment for x posts from 2022:  0.631410582646036
Number of posts from 2023:  69
Average sentiment for x posts from 2023:  0.5252912540561777
Number of posts from 2024:  513
Average sentiment for x posts from 2024:  0.38173854148801994
Number of

In [16]:
def sentiment_by_company(company1, company2=None):
    company_posts = posts[
        (posts[company1] == True) | (posts[company2] == True) if company2 else (posts[company1] == True)
    ]
    return avg_sentiment_calculation(company_posts)

print("Number of Amazon-related posts:", len(posts[posts["AWS"] == True]) + len(posts[posts["Amazon"] == True]))
print("Average sentiment of AWS-related posts:", sentiment_by_company("AWS", "Amazon"))
print("Number of Google-related posts:", len(posts[posts["Google"] == True]))
print("Average sentiment of Google-related posts:", sentiment_by_company("Google"))
print("Number of Microsoft-related posts:", len(posts[posts["Microsoft"] == True]) + len(posts[posts["Azure"] == True]))
print("Average sentiment of Microsoft-related posts:", sentiment_by_company("Microsoft", "Azure"))
print("Number of Meta-related posts:", len(posts[posts["Meta"] == True]) + len(posts[posts["Facebook"] == True]))
print("Average sentiment of Meta-related posts:", sentiment_by_company("Meta"))
print("Number of Oracle-related posts:", len(posts[posts["Oracle"] == True]))
print("Average sentiment of Oracle-related posts:", sentiment_by_company("Oracle"))
print("Number of Equinix-related posts:", len(posts[posts["Equinix"] == True]))
print("Average sentiment of Equinix-related posts:", sentiment_by_company("Equinix"))
print("Number of Digital Realty-related posts:", len(posts[posts["Digital Realty"] == True]))
print("Average sentiment of Digital Realty-related posts:", sentiment_by_company("Digital Realty"))
print("Number of IBM-related posts:", len(posts[posts["IBM"] == True]))
print("Average sentiment of IBM-related posts:", sentiment_by_company("IBM"))
print("Number of Apple-related posts:", len(posts[posts["Apple"] == True]))
print("Average sentiment of Apple-related posts:", sentiment_by_company("Apple"))
print("Number of QTS-related posts:", len(posts[posts["QTS"] == True]))
print("Average sentiment of QTS-related posts:", sentiment_by_company("QTS"))
print("Number of Vantage-related posts:", len(posts[posts["Vantage"] == True]))
print("Average sentiment of Vantage-related posts:", sentiment_by_company("Vantage"))
print("Number of CyrusOne-related posts:", len(posts[posts["CyrusOne"] == True]))
print("Average sentiment of CyrusOne-related posts:", sentiment_by_company("CyrusOne"))
print("Number of CoreSite-related posts:", len(posts[posts["CoreSite"] == True]))
print("Average sentiment of CoreSite-related posts:", sentiment_by_company("CoreSite"))

Number of Amazon-related posts: 419
Average sentiment of AWS-related posts: 0.24594983613070734
Number of Google-related posts: 449
Average sentiment of Google-related posts: 0.2258897837624088
Number of Microsoft-related posts: 488
Average sentiment of Microsoft-related posts: 0.29364274835271736
Number of Meta-related posts: 413
Average sentiment of Meta-related posts: 0.18685672770458883
Number of Oracle-related posts: 228
Average sentiment of Oracle-related posts: 0.2689305925752665
Number of Equinix-related posts: 22
Average sentiment of Equinix-related posts: 0.40289550840243266
Number of Digital Realty-related posts: 9
Average sentiment of Digital Realty-related posts: 0.45721611683354896
Number of IBM-related posts: 30
Average sentiment of IBM-related posts: 0.224594457304884
Number of Apple-related posts: 105
Average sentiment of Apple-related posts: 0.0561021903256601
Number of QTS-related posts: 8
Average sentiment of QTS-related posts: 0.5465720864573755
Number of Vantage-r

In [17]:
def analysis_of_viral_posts(min):
    viral_posts = posts.loc[posts["number of views"] >= min]
    print("What's considered viral: posts with over", min, "number of views")
    print("Number of viral posts:", len(viral_posts))
    print("Average sentiment of viral posts:", avg_sentiment_calculation(viral_posts))
    print("Average sentiment of non-viral posts:", avg_sentiment_calculation(posts.loc[posts["number of views"] < min]))
    print("Amount percent of viral posts that are polar (degree) > 0.70: ", len(viral_posts.loc[viral_posts["degree"] > 0.70])/len(viral_posts))

analysis_of_viral_posts(100)
print("\n")
analysis_of_viral_posts(500)

What's considered viral: posts with over 100 number of views
Number of viral posts: 4437
Average sentiment of viral posts: 0.31192107226259014
Average sentiment of non-viral posts: 0.4587031159102829
Amount percent of viral posts that are polar (degree) > 0.70:  0.5418075276087446


What's considered viral: posts with over 500 number of views
Number of viral posts: 3787
Average sentiment of viral posts: 0.29581112155197997
Average sentiment of non-viral posts: 0.4362556461122822
Amount percent of viral posts that are polar (degree) > 0.70:  0.5447583839450753
